In [1]:
import torch
from torchinfo import summary
from torch.utils.data import Dataset, DataLoader, random_split
import pandas as pd
import os
from dotenv import load_dotenv
import cv2
import glob
from random import randint
import lightning as L

from torch.nn.utils.rnn import pad_sequence

from src.transformer.transformer_architecture import KeypointTransformer, LitKeypointTransformer
from src.image_processing.mediapipe import MediaPipeProcessor

In [2]:
load_dotenv()
GESTURE_VIDEOS_PATH = os.getenv("GESTURE_VIDEOS")

In [3]:
# Dataset
df = pd.read_csv(f"{GESTURE_VIDEOS_PATH}/metadata.csv")
df.head()

,Video Name,Frames,Sex,Hand,Background,Illumination,People in Scene,Background Motion,Set,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16,Unnamed: 17
0,1CM1_4_R__229,3751,W,Right,Clutter,Stable,Single,Static,train,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1CM1_4_R__230,3684,W,Right,Clutter,Stable,Single,Static,train,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1CM1_4_R__231,3747,W,Right,Clutter,Stable,Single,Static,train,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1CM1_4_R__232,3858,W,Right,Clutter,Stable,Single,Static,train,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1CM42_11_R__205,3686,M,Right,Plain,Stable,Single,Static,train,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
file = open(f"{GESTURE_VIDEOS_PATH}/annotations/annotations/class_details.txt")
text = file.read().split('\n')
label_id_dict = dict()
for row in text:
    info = row.split('\t')[:3]
    label_id_dict[info[0]] = info[1:] 

In [5]:
label_id_dict

{'id': ['Label', 'Gesture'],
 '1': ['D0X', 'Non-gesture'],
 '2': ['B0A', 'Pointing with one finger'],
 '3': ['B0B', 'Pointing with two fingers'],
 '4': ['G01', 'Click with one finger'],
 '5': ['G02', 'Click with two fingers'],
 '6': ['G03', 'Throw up'],
 '7': ['G04', 'Throw down'],
 '8': ['G05', 'Throw left'],
 '9': ['G06', 'Throw right'],
 '10': ['G07', 'Open twice'],
 '11': ['G08', 'Double click with one finger'],
 '12': ['G09', 'Double click with two fingers'],
 '13': ['G10', 'Zoom in'],
 '14': ['G11', 'Zoom out'],
 '': []}

In [6]:
file = open(f"{GESTURE_VIDEOS_PATH}/annotations/annotations/Annot_List.txt")
text = file.read().split('\n')
video_info_dict = dict()
for row in text:
    info = row.split(',')
    video_info_dict[info[0]] = info[1:] 

In [7]:
video_info_dict

{'video': ['label', 'id', 't_start', 't_end', 'frames'],
 '1CM1_4_R__229': ['D0X', '1', '3677', '3751', '75'],
 '1CM1_4_R__230': ['D0X', '1', '3633', '3684', '52'],
 '1CM1_4_R__231': ['D0X', '1', '3690', '3747', '58'],
 '1CM1_4_R__232': ['D0X', '1', '3810', '3858', '49'],
 '1CM42_11_R__205': ['G05', '8', '3658', '3686', '29'],
 '1CM42_11_R__206': ['D0X', '1', '4654', '4682', '29'],
 '1CM42_11_R__207': ['G11', '14', '3641', '3682', '42'],
 '1CM42_11_R__208': ['G05', '8', '3639', '3664', '26'],
 '1CM42_12_R__157': ['D0X', '1', '4241', '4287', '47'],
 '1CM42_12_R__158': ['B0B', '3', '4395', '4476', '82'],
 '1CM42_12_R__159': ['D0X', '1', '4307', '4383', '77'],
 '1CM42_12_R__160': ['B0A', '2', '4369', '4434', '66'],
 '1CM42_13_R__141': ['D0X', '1', '3637', '3687', '51'],
 '1CM42_13_R__142': ['D0X', '1', '3660', '3747', '88'],
 '1CM42_13_R__143': ['D0X', '1', '3732', '3786', '55'],
 '1CM42_13_R__144': ['D0X', '1', '3814', '3859', '46'],
 '1CM42_17_R__189': ['D0X', '1', '3558', '3616', '59']

In [8]:
# Correct info dicts
new_label_id_dict = {}
new_video_info_dict = {}

for k, v in label_id_dict.items():
    if k.isdigit() and len(v) > 0:
        new_label_id_dict[f"{int(k) - 1}"] = v

label_id_dict = new_label_id_dict

for k, v in video_info_dict.items():
    label, id, t_start, t_end, frames = v
    if id.isdigit():
        new_video_info_dict[k] = [label, f"{int(id) - 1}", t_start, t_end, frames]

video_info_dict = new_video_info_dict

In [9]:
label_id_dict

{'0': ['D0X', 'Non-gesture'],
 '1': ['B0A', 'Pointing with one finger'],
 '2': ['B0B', 'Pointing with two fingers'],
 '3': ['G01', 'Click with one finger'],
 '4': ['G02', 'Click with two fingers'],
 '5': ['G03', 'Throw up'],
 '6': ['G04', 'Throw down'],
 '7': ['G05', 'Throw left'],
 '8': ['G06', 'Throw right'],
 '9': ['G07', 'Open twice'],
 '10': ['G08', 'Double click with one finger'],
 '11': ['G09', 'Double click with two fingers'],
 '12': ['G10', 'Zoom in'],
 '13': ['G11', 'Zoom out']}

In [10]:
video_info_dict

{'1CM1_4_R__229': ['D0X', '0', '3677', '3751', '75'],
 '1CM1_4_R__230': ['D0X', '0', '3633', '3684', '52'],
 '1CM1_4_R__231': ['D0X', '0', '3690', '3747', '58'],
 '1CM1_4_R__232': ['D0X', '0', '3810', '3858', '49'],
 '1CM42_11_R__205': ['G05', '7', '3658', '3686', '29'],
 '1CM42_11_R__206': ['D0X', '0', '4654', '4682', '29'],
 '1CM42_11_R__207': ['G11', '13', '3641', '3682', '42'],
 '1CM42_11_R__208': ['G05', '7', '3639', '3664', '26'],
 '1CM42_12_R__157': ['D0X', '0', '4241', '4287', '47'],
 '1CM42_12_R__158': ['B0B', '2', '4395', '4476', '82'],
 '1CM42_12_R__159': ['D0X', '0', '4307', '4383', '77'],
 '1CM42_12_R__160': ['B0A', '1', '4369', '4434', '66'],
 '1CM42_13_R__141': ['D0X', '0', '3637', '3687', '51'],
 '1CM42_13_R__142': ['D0X', '0', '3660', '3747', '88'],
 '1CM42_13_R__143': ['D0X', '0', '3732', '3786', '55'],
 '1CM42_13_R__144': ['D0X', '0', '3814', '3859', '46'],
 '1CM42_17_R__189': ['D0X', '0', '3558', '3616', '59'],
 '1CM42_17_R__190': ['D0X', '0', '3450', '3601', '152']

In [11]:
num_videos = len(glob.glob(f'{GESTURE_VIDEOS_PATH}/videos/videos/*'))
print(len(video_info_dict))
print(num_videos)

200
200


In [12]:
class IPNData(Dataset):
    def __init__(self, video_info):
        self.paths = glob.glob(f'{GESTURE_VIDEOS_PATH}/videos/videos/*')
        self.video_info = video_info
        
    def get_frames(self, path, info):
        frames = []
        frame_no = 1
        label, ID, start_frame, end_frame, num_frames = info
        cap = cv2.VideoCapture(str(path))
        cap.set(cv2.CAP_PROP_FPS, 5)
        ret, frame = cap.read()

        start_frame = int(start_frame)
        end_frame = int(end_frame)

        # Loop over the frames
        while ret:

            if start_frame <= frame_no < end_frame:
                frames.append(frame)

            frame_no += 1


            # Read the next frame
            ret, frame = cap.read()

        # Release the video capture
        cap.release()
        return frames
    
    def __getitem__(self, ix):
        path = self.paths[ix % len(self.paths)]
        video_name = str(path).split('/')[-1][:-4]
        info = self.video_info[video_name]
        label, ID, start_frame, end_frame, num_frames = info
        frames = self.get_frames(path, info)
        return frames, label, ID
    
    def __len__(self):
        return len(self.paths)
    
    def choose(self): return self[randint(len(self))]

In [13]:
ds_raw = IPNData(video_info=video_info_dict)
mpp = MediaPipeProcessor()

INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1772839385.557168   42264 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1772839385.564877   42264 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


In [14]:
label_to_index = {}

for k in label_id_dict:
    label, gesture = label_id_dict[k]
    label_to_index[label] = int(k)

In [ ]:
N = len(video_info_dict)
T = 1
ds = []
train_size = int(0.9 * (N - T))
val_size = (N -T) - train_size

for i in range(N):
    frames, label, ID = ds_raw.__getitem__(i)

    keypoint_sequence = mpp.process_video(frames)

    ds.append([keypoint_sequence, label_to_index[label]]) # using ID as label

test_ds = ds[-1]
train_val_ds = ds[:-1]

train_ds, val_ds = random_split(train_val_ds, [train_size, val_size])

W0000 00:00:1772839385.921371   42268 landmark_projection_calculator.cc:78] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSIONS or use PROJECTION_MATRIX.


In [ ]:
print(len(train_ds))
print(len(val_ds))

180
20


In [ ]:
print(test_ds)

[array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], shape=(53, 126)), 1]


In [ ]:
def collate_fn(batch):
    sequences, labels = zip(*batch)

    sequences = [torch.tensor(s, dtype=torch.float32) for s in sequences]

    # Padding
    padded_sequences = pad_sequence(sequences, batch_first=True)

    # Masking
    mask = torch.zeros(size=padded_sequences.shape[:2], dtype=torch.bool)
    for(i, s) in enumerate(sequences):
        mask[i, :len(s)] = True

    # Labeling
    labels = torch.tensor([int(l) for l in labels])

    return padded_sequences, mask, labels

In [ ]:
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_ds, batch_size=32, collate_fn=collate_fn)

input_size = 126
d_model = 132

num_classes = len(label_id_dict[1:-1])

print(num_classes)

model = LitKeypointTransformer(input_size=input_size, d_model=d_model, num_classes=num_classes)

trainer = L.Trainer(
    max_epochs=20,
    accelerator="auto",
    devices=1
)


KeyError: slice(1, -1, None)

In [ ]:
x_batch, mask, y_batch = next(iter(train_loader))
logits = model(x_batch)  
print("x:", x_batch.shape, "y:", y_batch.shape, "logits:", logits.shape)

x: torch.Size([32, 372, 126]) y: torch.Size([32]) logits: torch.Size([32, 16])


In [ ]:
trainer.fit(model, train_loader, val_loader)

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
You are using a CUDA device ('NVIDIA GeForce RTX 3070') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name    | Type                | Params | Mode  | FLOPs
----------------------------------------------------------------
0 | model   | KeypointTransformer | 1.3 M  | train | 0    
1 | loss_fn | CrossEntropyLoss    | 0      | train | 0    
----------------------------------------------------------------
1.3 M     Trainable params
0         Non-trainable params
1.3 M     Total params


Sanity Checking DataLoader 0:   0%|          | 0/1 [00:00<?, ?it/s]

/home/mark-oliver/git/gesture_recognition/.venv/lib/python3.12/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/home/mark-oliver/git/gesture_recognition/.venv/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.


/home/mark-oliver/git/gesture_recognition/.venv/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.
/home/mark-oliver/git/gesture_recognition/.venv/lib/python3.12/site-packages/lightning/pytorch/loops/fit_loop.py:317: The number of training batches (6) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if you want to see logs for the training epoch.


Epoch 19:  67%|██████▋   | 4/6 [00:15<00:07,  0.25it/s, v_num=12, train_loss_step=0.765, val_loss=1.160, train_loss_epoch=0.645]

In [ ]:
model.eval()

LitKeypointTransformer(
  (model): KeypointTransformer(
    (input_proj): Sequential(
      (0): Linear(in_features=126, out_features=132, bias=True)
    )
    (pos_encode): SinusoidalPositionalEncoding()
    (layers): ModuleList(
      (0-5): 6 x EncodingLayer(
        (attention): MultiHeadAttention(
          (w_q): Linear(in_features=132, out_features=132, bias=True)
          (w_k): Linear(in_features=132, out_features=132, bias=True)
          (w_v): Linear(in_features=132, out_features=132, bias=True)
          (output): Linear(in_features=132, out_features=132, bias=True)
        )
        (norm1): LayerNorm((132,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((132,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
        (ffn): Sequential(
          (0): Linear(in_features=132, out_features=512, bias=True)
          (1): GELU(approximate='none')
          (2): Linear(in_f

In [ ]:
#Testing model

In [ ]:
test_ds[0]

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], shape=(53, 126))

In [ ]:
keypoint_sequence, label = test_ds

x = torch.tensor(keypoint_sequence, dtype=torch.float32).unsqueeze(0)  # (1, seq_len, 126)

# prediction
with torch.no_grad():
    logits = model(x)
    predicted_idx = torch.argmax(logits, dim=-1).item()

print(f"Prediction: {label_id_dict[f"{predicted_idx}"]}")
print(f"Reality: {label_id_dict[f"{label}"]}")

Prediction: ['D0X', 'Non-gesture']
Reality: ['D0X', 'Non-gesture']


In [ ]:
for i in range(150):    
    keypoint_sequence, label = train_ds[i]

    x = torch.tensor(keypoint_sequence, dtype=torch.float32).unsqueeze(0)  # (1, seq_len, 126)

    # prediction
    with torch.no_grad():
        logits = model(x)
        predicted_idx = torch.argmax(logits, dim=-1).item()

    print(f"Prediction: {predicted_idx}")
    print(f"Reality: {label}")

Prediction: 1
Reality: 1
Prediction: 1
Reality: 1
Prediction: 1
Reality: 1
Prediction: 1
Reality: 1
Prediction: 1
Reality: 1
Prediction: 1
Reality: 1
Prediction: 1
Reality: 1
Prediction: 1
Reality: 1
Prediction: 1
Reality: 1
Prediction: 1
Reality: 1
Prediction: 1
Reality: 1
Prediction: 1
Reality: 1
Prediction: 1
Reality: 1
Prediction: 1
Reality: 8
Prediction: 1
Reality: 1
Prediction: 1
Reality: 1
Prediction: 11
Reality: 11
Prediction: 1
Reality: 1
Prediction: 1
Reality: 1
Prediction: 1
Reality: 1
Prediction: 1
Reality: 1
Prediction: 1
Reality: 1
Prediction: 1
Reality: 1
Prediction: 1
Reality: 1
Prediction: 1
Reality: 1
Prediction: 1
Reality: 1
Prediction: 1
Reality: 1
Prediction: 1
Reality: 1
Prediction: 1
Reality: 4
Prediction: 1
Reality: 1
Prediction: 1
Reality: 1
Prediction: 1
Reality: 7
Prediction: 1
Reality: 1
Prediction: 1
Reality: 1
Prediction: 1
Reality: 1
Prediction: 1
Reality: 1
Prediction: 1
Reality: 2
Prediction: 1
Reality: 1
Prediction: 1
Reality: 1
Prediction: 1
Reality: 